In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
df = pd.read_json("/content/drive/MyDrive/cleaned_dataset.json")

In [27]:
print("="*80)
print("LIGHTWEIGHT FEATURE ENGINEERING (NO ONE-HOT ENCODING)")
print("="*80)

columns_to_drop = [
    'success', 'Title', 'Movie Link', 'description',
    'Rating', 'Votes'
]

X_light = df.drop(columns=columns_to_drop).copy()
y = df['success']

print(f"Starting features: {X_light.shape[1]}")
print(f"Features: {list(X_light.columns)}")

LIGHTWEIGHT FEATURE ENGINEERING (NO ONE-HOT ENCODING)
Starting features: 12
Features: ['Year', 'Duration', 'MPA', 'writers', 'directors', 'stars', 'countries_origin', 'filming_locations', 'production_company', 'genres', 'Languages', 'release_decade']


In [28]:
# Keep numerical features as-is
numerical_features = ['Year', 'Duration', 'release_decade']

# ALL categorical features will use COUNT ENCODING
categorical_features = [
    'MPA', 'writers', 'directors', 'stars', 'countries_origin',
    'filming_locations', 'production_company', 'genres', 'Languages'
]

print(f"\n📊 Numerical features: {numerical_features}")
print(f"📊 Categorical features (will use count encoding): {categorical_features}")


📊 Numerical features: ['Year', 'Duration', 'release_decade']
📊 Categorical features (will use count encoding): ['MPA', 'writers', 'directors', 'stars', 'countries_origin', 'filming_locations', 'production_company', 'genres', 'Languages']


In [29]:
print("="*80)
print("APPLYING COUNT ENCODING TO ALL CATEGORICAL FEATURES")
print("="*80)

X_final = X_light.copy()

for col in categorical_features:
    print(f"Processing {col}...")

    # IMPORTANT: Convert lists/tuples to strings first
    X_final[col] = X_final[col].astype(str)

    # Now count how many times each string value appears
    value_counts = X_final[col].value_counts()

    # Replace each value with its count
    X_final[col] = X_final[col].map(value_counts)

    print(f"  ✓ {col}: encoded to counts")

print(f"\n✅ Final dataset shape: {X_final.shape}")
print(f"✅ Features: {list(X_final.columns)}")

APPLYING COUNT ENCODING TO ALL CATEGORICAL FEATURES
Processing MPA...
  ✓ MPA: encoded to counts
Processing writers...
  ✓ writers: encoded to counts
Processing directors...
  ✓ directors: encoded to counts
Processing stars...
  ✓ stars: encoded to counts
Processing countries_origin...
  ✓ countries_origin: encoded to counts
Processing filming_locations...
  ✓ filming_locations: encoded to counts
Processing production_company...
  ✓ production_company: encoded to counts
Processing genres...
  ✓ genres: encoded to counts
Processing Languages...
  ✓ Languages: encoded to counts

✅ Final dataset shape: (62569, 12)
✅ Features: ['Year', 'Duration', 'MPA', 'writers', 'directors', 'stars', 'countries_origin', 'filming_locations', 'production_company', 'genres', 'Languages', 'release_decade']


In [30]:
# Verify - should be only 12 features now!
print("\n" + "="*80)
print("FINAL FEATURE SET")
print("="*80)

print(f"Total features: {X_final.shape[1]}")
print(f"Dataset shape: {X_final.shape}")
print(f"\nFeature list:")
for i, col in enumerate(X_final.columns, 1):
    print(f"  {i}. {col}")

print(f"\nMemory usage: {X_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


FINAL FEATURE SET
Total features: 12
Dataset shape: (62569, 12)

Feature list:
  1. Year
  2. Duration
  3. MPA
  4. writers
  5. directors
  6. stars
  7. countries_origin
  8. filming_locations
  9. production_company
  10. genres
  11. Languages
  12. release_decade

Memory usage: 5.73 MB


In [31]:
from sklearn.model_selection import train_test_split

# Train-Test Split
print("\n" + "="*80)
print("TRAIN-TEST SPLIT")
print("="*80)

X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_final, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"✓ Training set: {X_train_final.shape}")
print(f"✓ Test set: {X_test_final.shape}")
print(f"\n✓ Ready to train models! 🚀")

# Initialize results storage
results_final = []


TRAIN-TEST SPLIT
✓ Training set: (50055, 12)
✓ Test set: (12514, 12)

✓ Ready to train models! 🚀


In [33]:
from sklearn.preprocessing import StandardScaler

print("="*80)
print("FEATURE SCALING")
print("="*80)

scaler_final = StandardScaler()
X_train_scaled_final = scaler_final.fit_transform(X_train_final)
X_test_scaled_final = scaler_final.transform(X_test_final)

print(f"✓ Features scaled using StandardScaler")
print(f"✓ Training set shape: {X_train_scaled_final.shape}")
print(f"✓ Test set shape: {X_test_scaled_final.shape}")

FEATURE SCALING
✓ Features scaled using StandardScaler
✓ Training set shape: (50055, 12)
✓ Test set shape: (12514, 12)


In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print("\n" + "="*80)
print("MODEL 1: LOGISTIC REGRESSION")
print("="*80)

lr_model = LogisticRegression(max_iter=1000, random_state=42)

print("Training...")
lr_model.fit(X_train_scaled_final, y_train_final)

y_pred_lr = lr_model.predict(X_test_scaled_final)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled_final)[:, 1]

lr_accuracy = accuracy_score(y_test_final, y_pred_lr)
lr_precision = precision_score(y_test_final, y_pred_lr, average='weighted')
lr_recall = recall_score(y_test_final, y_pred_lr, average='weighted')
lr_f1 = f1_score(y_test_final, y_pred_lr, average='weighted')
lr_roc_auc = roc_auc_score(y_test_final, y_pred_proba_lr)

results_final.append({
    'Model': 'Logistic Regression',
    'Accuracy': lr_accuracy,
    'Precision': lr_precision,
    'Recall': lr_recall,
    'F1-Score': lr_f1,
    'ROC-AUC': lr_roc_auc
})

print(f"\n✓ Training completed!")
print(f"\nMetrics:")
print(f"  Accuracy:  {lr_accuracy:.4f}")
print(f"  Precision: {lr_precision:.4f}")
print(f"  Recall:    {lr_recall:.4f}")
print(f"  F1-Score:  {lr_f1:.4f}")
print(f"  ROC-AUC:   {lr_roc_auc:.4f}")

cm_lr = confusion_matrix(y_test_final, y_pred_lr)
print(f"\nConfusion Matrix:")
print(cm_lr)


MODEL 1: LOGISTIC REGRESSION
Training...

✓ Training completed!

Metrics:
  Accuracy:  0.8569
  Precision: 0.8235
  Recall:    0.8569
  F1-Score:  0.8231
  ROC-AUC:   0.8256

Confusion Matrix:
[[10414   259]
 [ 1532   309]]


In [35]:
from sklearn.tree import DecisionTreeClassifier

print("\n" + "="*80)
print("MODEL 2: DECISION TREE")
print("="*80)

dt_model = DecisionTreeClassifier(random_state=42, max_depth=10)

print("Training...")
dt_model.fit(X_train_final, y_train_final)  # Decision Tree doesn't need scaling

y_pred_dt = dt_model.predict(X_test_final)
y_pred_proba_dt = dt_model.predict_proba(X_test_final)[:, 1]

dt_accuracy = accuracy_score(y_test_final, y_pred_dt)
dt_precision = precision_score(y_test_final, y_pred_dt, average='weighted')
dt_recall = recall_score(y_test_final, y_pred_dt, average='weighted')
dt_f1 = f1_score(y_test_final, y_pred_dt, average='weighted')
dt_roc_auc = roc_auc_score(y_test_final, y_pred_proba_dt)

results_final.append({
    'Model': 'Decision Tree',
    'Accuracy': dt_accuracy,
    'Precision': dt_precision,
    'Recall': dt_recall,
    'F1-Score': dt_f1,
    'ROC-AUC': dt_roc_auc
})

print(f"\n✓ Training completed!")
print(f"\nMetrics:")
print(f"  Accuracy:  {dt_accuracy:.4f}")
print(f"  Precision: {dt_precision:.4f}")
print(f"  Recall:    {dt_recall:.4f}")
print(f"  F1-Score:  {dt_f1:.4f}")
print(f"  ROC-AUC:   {dt_roc_auc:.4f}")

cm_dt = confusion_matrix(y_test_final, y_pred_dt)
print(f"\nConfusion Matrix:")
print(cm_dt)

print(f"\nTop 10 Most Important Features:")
feature_importance_dt = pd.DataFrame({
    'Feature': X_train_final.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance_dt.head(10))


MODEL 2: DECISION TREE
Training...

✓ Training completed!

Metrics:
  Accuracy:  0.8660
  Precision: 0.8495
  Recall:    0.8660
  F1-Score:  0.8542
  ROC-AUC:   0.8327

Confusion Matrix:
[[10146   527]
 [ 1150   691]]

Top 10 Most Important Features:
               Feature  Importance
1             Duration    0.317765
9               genres    0.196746
7    filming_locations    0.087961
0                 Year    0.074299
2                  MPA    0.067417
10           Languages    0.059007
11      release_decade    0.051881
8   production_company    0.048237
3              writers    0.034387
6     countries_origin    0.025038


In [19]:
from sklearn.ensemble import RandomForestClassifier

print("\n" + "="*80)
print("MODEL 3: RANDOM FOREST")
print("="*80)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, n_jobs=-1)

print("Training...")
rf_model.fit(X_train_final, y_train_final)  # Random Forest doesn't need scaling

y_pred_rf = rf_model.predict(X_test_final)
y_pred_proba_rf = rf_model.predict_proba(X_test_final)[:, 1]

rf_accuracy = accuracy_score(y_test_final, y_pred_rf)
rf_precision = precision_score(y_test_final, y_pred_rf, average='weighted')
rf_recall = recall_score(y_test_final, y_pred_rf, average='weighted')
rf_f1 = f1_score(y_test_final, y_pred_rf, average='weighted')
rf_roc_auc = roc_auc_score(y_test_final, y_pred_proba_rf)

results_final.append({
    'Model': 'Random Forest',
    'Accuracy': rf_accuracy,
    'Precision': rf_precision,
    'Recall': rf_recall,
    'F1-Score': rf_f1,
    'ROC-AUC': rf_roc_auc
})

print(f"\n✓ Training completed!")
print(f"\nMetrics:")
print(f"  Accuracy:  {rf_accuracy:.4f}")
print(f"  Precision: {rf_precision:.4f}")
print(f"  Recall:    {rf_recall:.4f}")
print(f"  F1-Score:  {rf_f1:.4f}")
print(f"  ROC-AUC:   {rf_roc_auc:.4f}")

cm_rf = confusion_matrix(y_test_final, y_pred_rf)
print(f"\nConfusion Matrix:")
print(cm_rf)

print(f"\nTop 10 Most Important Features:")
feature_importance_rf = pd.DataFrame({
    'Feature': X_train_final.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance_rf.head(10))


MODEL 3: RANDOM FOREST
Training...

✓ Training completed!

Metrics:
  Accuracy:  0.8762
  Precision: 0.8594
  Recall:    0.8762
  F1-Score:  0.8531
  ROC-AUC:   0.8843

Confusion Matrix:
[[10445   228]
 [ 1321   520]]

Top 10 Most Important Features:
               Feature  Importance
1             Duration    0.251311
9               genres    0.155679
0                 Year    0.104761
7    filming_locations    0.101619
10           Languages    0.080756
11      release_decade    0.073544
2                  MPA    0.066337
8   production_company    0.058718
6     countries_origin    0.037905
3              writers    0.031131


In [20]:
from sklearn.neighbors import KNeighborsClassifier

print("\n" + "="*80)
print("MODEL 4: K-NEAREST NEIGHBORS")
print("="*80)

knn_model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

print("Training...")
knn_model.fit(X_train_scaled_final, y_train_final)  # KNN needs scaling

y_pred_knn = knn_model.predict(X_test_scaled_final)
y_pred_proba_knn = knn_model.predict_proba(X_test_scaled_final)[:, 1]

knn_accuracy = accuracy_score(y_test_final, y_pred_knn)
knn_precision = precision_score(y_test_final, y_pred_knn, average='weighted')
knn_recall = recall_score(y_test_final, y_pred_knn, average='weighted')
knn_f1 = f1_score(y_test_final, y_pred_knn, average='weighted')
knn_roc_auc = roc_auc_score(y_test_final, y_pred_proba_knn)

results_final.append({
    'Model': 'K-Nearest Neighbors',
    'Accuracy': knn_accuracy,
    'Precision': knn_precision,
    'Recall': knn_recall,
    'F1-Score': knn_f1,
    'ROC-AUC': knn_roc_auc
})

print(f"\n✓ Training completed!")
print(f"\nMetrics:")
print(f"  Accuracy:  {knn_accuracy:.4f}")
print(f"  Precision: {knn_precision:.4f}")
print(f"  Recall:    {knn_recall:.4f}")
print(f"  F1-Score:  {knn_f1:.4f}")
print(f"  ROC-AUC:   {knn_roc_auc:.4f}")

cm_knn = confusion_matrix(y_test_final, y_pred_knn)
print(f"\nConfusion Matrix:")
print(cm_knn)


MODEL 4: K-NEAREST NEIGHBORS
Training...

✓ Training completed!

Metrics:
  Accuracy:  0.8582
  Precision: 0.8365
  Recall:    0.8582
  F1-Score:  0.8423
  ROC-AUC:   0.7792

Confusion Matrix:
[[10165   508]
 [ 1266   575]]


In [21]:
from sklearn.naive_bayes import GaussianNB

print("\n" + "="*80)
print("MODEL 5: NAIVE BAYES")
print("="*80)

nb_model = GaussianNB()

print("Training...")
nb_model.fit(X_train_scaled_final, y_train_final)  # Naive Bayes with scaling

y_pred_nb = nb_model.predict(X_test_scaled_final)
y_pred_proba_nb = nb_model.predict_proba(X_test_scaled_final)[:, 1]

nb_accuracy = accuracy_score(y_test_final, y_pred_nb)
nb_precision = precision_score(y_test_final, y_pred_nb, average='weighted')
nb_recall = recall_score(y_test_final, y_pred_nb, average='weighted')
nb_f1 = f1_score(y_test_final, y_pred_nb, average='weighted')
nb_roc_auc = roc_auc_score(y_test_final, y_pred_proba_nb)

results_final.append({
    'Model': 'Naive Bayes',
    'Accuracy': nb_accuracy,
    'Precision': nb_precision,
    'Recall': nb_recall,
    'F1-Score': nb_f1,
    'ROC-AUC': nb_roc_auc
})

print(f"\n✓ Training completed!")
print(f"\nMetrics:")
print(f"  Accuracy:  {nb_accuracy:.4f}")
print(f"  Precision: {nb_precision:.4f}")
print(f"  Recall:    {nb_recall:.4f}")
print(f"  F1-Score:  {nb_f1:.4f}")
print(f"  ROC-AUC:   {nb_roc_auc:.4f}")

cm_nb = confusion_matrix(y_test_final, y_pred_nb)
print(f"\nConfusion Matrix:")
print(cm_nb)


MODEL 5: NAIVE BAYES
Training...

✓ Training completed!

Metrics:
  Accuracy:  0.7108
  Precision: 0.8319
  Recall:    0.7108
  F1-Score:  0.7494
  ROC-AUC:   0.7785

Confusion Matrix:
[[7672 3001]
 [ 618 1223]]


In [36]:
from sklearn.ensemble import GradientBoostingClassifier

print("\n" + "="*80)
print("MODEL 6: GRADIENT BOOSTING")
print("="*80)

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

# Train
print("Training Gradient Boosting (100 trees)...")
gb_model.fit(X_train_final, y_train_final)  # Doesn't need scaling

# Predict
y_pred_gb = gb_model.predict(X_test_final)
y_pred_proba_gb = gb_model.predict_proba(X_test_final)[:, 1]

# Calculate metrics
gb_accuracy = accuracy_score(y_test_final, y_pred_gb)
gb_precision = precision_score(y_test_final, y_pred_gb, average='weighted')
gb_recall = recall_score(y_test_final, y_pred_gb, average='weighted')
gb_f1 = f1_score(y_test_final, y_pred_gb, average='weighted')
gb_roc_auc = roc_auc_score(y_test_final, y_pred_proba_gb)

# Store results
results_final.append({
    'Model': 'Gradient Boosting',
    'Accuracy': gb_accuracy,
    'Precision': gb_precision,
    'Recall': gb_recall,
    'F1-Score': gb_f1,
    'ROC-AUC': gb_roc_auc
})

# Display
print(f"\n✓ Training completed!")
print(f"\nMetrics:")
print(f"  Accuracy:  {gb_accuracy:.4f}")
print(f"  Precision: {gb_precision:.4f}")
print(f"  Recall:    {gb_recall:.4f}")
print(f"  F1-Score:  {gb_f1:.4f}")
print(f"  ROC-AUC:   {gb_roc_auc:.4f}")

# Confusion Matrix
cm_gb = confusion_matrix(y_test_final, y_pred_gb)
print(f"\nConfusion Matrix:")
print(cm_gb)

# Feature importance
print(f"\nTop 10 Most Important Features:")
feature_importance_gb = pd.DataFrame({
    'Feature': X_train_final.columns,
    'Importance': gb_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance_gb.head(10))


MODEL 6: GRADIENT BOOSTING
Training Gradient Boosting (100 trees)...

✓ Training completed!

Metrics:
  Accuracy:  0.8805
  Precision: 0.8654
  Recall:    0.8805
  F1-Score:  0.8654
  ROC-AUC:   0.8896

Confusion Matrix:
[[10341   332]
 [ 1163   678]]

Top 10 Most Important Features:
               Feature  Importance
1             Duration    0.328549
9               genres    0.160876
2                  MPA    0.106319
7    filming_locations    0.086923
0                 Year    0.074135
10           Languages    0.070698
11      release_decade    0.050609
8   production_company    0.041641
3              writers    0.032126
6     countries_origin    0.024315


In [37]:
print("\n" + "="*80)
print("FINAL RESULTS - ALL MODELS COMPARISON")
print("="*80)

results_df_final = pd.DataFrame(results_final)
results_df_final = results_df_final.sort_values('F1-Score', ascending=False).reset_index(drop=True)

print("\n")
print(results_df_final.to_string(index=False))

print("\n" + "="*80)
print(f"BEST MODEL: {results_df_final.iloc[0]['Model']}")
print(f"   F1-Score: {results_df_final.iloc[0]['F1-Score']:.4f}")
print(f"   Accuracy: {results_df_final.iloc[0]['Accuracy']:.4f}")
print(f"   ROC-AUC:  {results_df_final.iloc[0]['ROC-AUC']:.4f}")
print("="*80)


FINAL RESULTS - ALL MODELS COMPARISON


              Model  Accuracy  Precision   Recall  F1-Score  ROC-AUC
  Gradient Boosting  0.880534   0.865419 0.880534  0.865361 0.889587
      Decision Tree  0.865990   0.849518 0.865990  0.854244 0.832718
Logistic Regression  0.856880   0.823540 0.856880  0.823093 0.825560

BEST MODEL: Gradient Boosting
   F1-Score: 0.8654
   Accuracy: 0.8805
   ROC-AUC:  0.8896
